In [ ]:
import os
import requests
import json
from pathlib import Path

In [ ]:
ACCESS_KEY = ""

IMAGES_DIR = Path("images")
DATA_DIR = Path("data")

IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

NB_IMAGES_PER_CATEGORY = 10

## Partie 1 : Récupération des images

A cause de la limite api de unspash nous récupérons uniquement 30 images

In [ ]:
HEADERS = {"Authorization": f"Client-ID {ACCESS_KEY}"}


CATEGORIES = {
    "animaux": "animals",
    "batiments": "architecture",
    "paysage": "landscape",
}

metadata_list = []

for category_name, query in CATEGORIES.items():
    print(f"Chargement de la catégorie {category_name}...")

    url = "https://api.unsplash.com/search/photos"
    params = {
        "query": query,
        "per_page": NB_IMAGES_PER_CATEGORY,
        "page": 1,
        "orientation": "landscape",
    }

    response = requests.get(url, headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    results = response.json()["results"]

    for i, photo in enumerate(results[:33], start=1):
        photo_id = photo["id"]

        url_photo = f"https://api.unsplash.com/photos/{photo_id}"
        response_photo = requests.get(url_photo, headers=HEADERS, timeout=30)
        response_photo.raise_for_status()
        photo_full = response_photo.json()

        # Nom du fichier
        filename = f"{category_name}_{photo_id}.jpg"
        filepath = IMAGES_DIR / filename

        # URL de l’image (regular ou full)
        image_url = photo_full["urls"]["regular"]

        # Téléchargement de l’image
        image_response = requests.get(image_url, stream=True, timeout=30)
        image_response.raise_for_status()

        with open(filepath, "wb") as f:
            for chunk in image_response.iter_content(chunk_size=8192):
                f.write(chunk)

        # Taille du fichier en Ko
        file_stat = filepath.stat()
        file_size_kb = round(file_stat.st_size / 1024, 2)

        # Metadonnées EXIF
        exif = photo_full.get("exif", {}) or {}
        make = exif.get("make") or ""
        model = exif.get("model") or ""
        camera = (make + " " + model).strip()
        aperture = exif.get("aperture")
        focal_length = exif.get("focal_length")
        iso = exif.get("iso")
        exposure = exif.get("exposure_time")

        taken_at = exif.get("created_at", "") or photo_full.get("created_at", "")

        licenses = photo_full.get("current_user_collections", [])
        license_str = "Unsplash license (check terms)"

        meta = {
            "nom_fichier": filename,
            "categorie": category_name,
            "largeur": photo_full["width"],
            "hauteur": photo_full["height"],
            "format_fichier": "jpg",
            "taille_fichier_kb": file_size_kb,
            "url_source": photo_full["urls"]["full"],
            "licence": license_str,
            "description": photo_full.get("description") or photo_full.get("alt_description"),
            "photographe": photo_full["user"]["name"],
            "page_unsplash": photo_full["links"]["html"],
            "date_creation_photo": photo_full["created_at"],
            "nombre_likes": photo_full.get("likes", 0),
            "exif": {
                "camera_model": camera if camera else None,
                "taken_at": taken_at if taken_at else None,
                "aperture": aperture,
                "focal_length": focal_length,
                "iso": iso,
                "exposure_time": exposure,
            },
        }

        metadata_list.append(meta)

# Sauvegarde des métadonnées
output_path = DATA_DIR / "images_metadata.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"✅ {len(metadata_list)} images et métadonnées sauvegardées.")

Chargement de la catégorie animaux...
Chargement de la catégorie batiments...
Chargement de la catégorie paysage...
✅ 9 images et métadonnées sauvegardées.


## Partie 2 : Annotation des images

In [6]:
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans

In [7]:
NUM_CLUSTERS = 5
DOWNSCALE_SIZE = 200 
MAX_SAMPLES = 10_000

In [8]:
def get_orientation(w, h):
    if abs(w - h) < 0.05 * min(w, h):  # quasiment carré
        return "carre"
    elif w > h:
        return "paysage"
    else:
        return "portrait"

In [9]:
def get_size_category(w, h):
    max_dim = max(w, h)
    if max_dim < 500:
        return "vignette"
    elif max_dim <= 1500:
        return "moyenne"
    else:
        return "grande"

In [10]:

def closest_color_name(rgb):
    r, g, b = rgb
    if r > 200 and g < 100 and b < 100:
        return "rouge"
    if r > 200 and g > 150 and b < 100:
        return "orange"
    if r < 100 and g > 150 and b < 100:
        return "vert"
    if r < 100 and g < 100 and b > 150:
        return "bleu"
    if r > 220 and g > 220 and b > 220:
        return "blanc"
    if r < 50 and g < 50 and b < 50:
        return "noir"
    if r > 150 and g > 150 and b > 150:
        return "gris clair"
    if r > 80 and g > 80 and b > 80:
        return "gris moyen"
    if r > 40 and g > 40 and b < 40:
        return "brun"
    return "autre"

In [11]:
def get_dominant_colors(img_path, num_clusters=NUM_CLUSTERS, max_samples=MAX_SAMPLES):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    # Réduction pour échantillonner moins de pixels
    if w > DOWNSCALE_SIZE or h > DOWNSCALE_SIZE:
        if w > h:
            new_w = DOWNSCALE_SIZE
            new_h = int(h * DOWNSCALE_SIZE / w)
        else:
            new_h = DOWNSCALE_SIZE
            new_w = int(w * DOWNSCALE_SIZE / h)
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

    numarray = np.array(img, dtype=np.uint8).reshape(-1, 3)

    # KMeans sur un échantillon réduit
    kmeans = KMeans(n_clusters=num_clusters, n_init=2, random_state=42)
    kmeans.fit(numarray)

    labels = kmeans.labels_
    counts = np.bincount(labels, minlength=num_clusters)
    sorted_indices = np.argsort(counts)[::-1]
    centroids = kmeans.cluster_centers_

    dominant_colors = []
    for idx in sorted_indices[:num_clusters]:
        r, g, b = np.clip(centroids[idx], 0, 255)
        dominant_colors.append([int(r), int(g), int(b)])

    return dominant_colors

In [13]:
# Parcours des images
labels_dict = {}

for img_path in IMAGES_DIR.iterdir():
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    dominant_colors_rgb = get_dominant_colors(img_path, num_clusters=5)
    color_names = [closest_color_name(rgb) for rgb in dominant_colors_rgb]

    orientation = get_orientation(w, h)
    size_category = get_size_category(w, h)

    tags = []

    filename = os.path.basename(img_path)

    labels_dict[filename] = {
        "predominant_colors": dominant_colors_rgb,
        "color_names": color_names,
        "orientation": orientation,
        "size_category": size_category,
        "tags": tags,
    }

# Sauvegarde du JSON
output_path = DATA_DIR / "images_labels.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labels_dict, f, ensure_ascii=False, indent=2)

print(f"Labels sauvegardés pour {len(labels_dict)} images dans {output_path}.")

Labels sauvegardés pour 9 images dans data\images_labels.json.
